In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

In [2]:
players_details = []

teams = ['DC','GG','MI','RCB','UPW']

for t in teams:
        # ব্রাউজার ব্যাকগ্রাউন্ডে চালানোর জন্য (হেডলেস মোড)
        options = Options()
        # options.add_argument("--headless")  # ব্রাউজার পপআপ না দেখতে চাইলে এটা অন করতে পারো
        
        driver = webdriver.Chrome(options=options)
        
        url = f"https://www.wplt20.com/auction/2026/teams?team={t}"
        driver.get(url)
        
        # পেজ এবং জাভাস্ক্রিপ্ট পুরোপুরি লোড হওয়ার জন্য ৩-৫ সেকেন্ড অপেক্ষা
        time.sleep(4)
        
        # এখন পেজের রেন্ডার হওয়া ফুল HTML সোর্স নিয়ে নেব
        html_content = driver.page_source
        driver.quit()
        
        # এবার BeautifulSoup দিয়ে আগের মতোই পার্স করো!
        soup = BeautifulSoup(html_content, "html.parser")
        
        team = soup.find('h2',class_='text-heading-6 truncate font-bold tablet:text-heading-4').text
        ammount = soup.find('p',class_='text-label-l2 font-extrabold tablet:text-heading-5').text.split()
        spent = float(ammount[0][1:])
        bought = int(soup.find_all('p',class_='text-label-l2 font-extrabold tablet:text-heading-5')[1].text)
        overseas = int(soup.find_all('p',class_='text-label-l2 font-extrabold tablet:text-heading-5')[2].text)
        rem = soup.find_all('p',class_='text-label-l2 font-extrabold tablet:text-heading-5')[3].text.split()
        remaining = float(rem[0][1:])
        
        team_details = {
            'team' : team,
            'spent(Cr)': spent,
            'bought':bought,
            'overseas':overseas,
            'remaining(Cr)':remaining,
        }
        
        fore = '<svg aria-hidden="true" class="text-content-accent size-4 shrink-0"'
        
        div = soup.find_all('tr',class_="border-regular text-label-l2 text-content-primary border-b last:border-b-0")
        
        c=0 
        p = len(div)//2 + len(div)%2
        
        for i in div :
            div2 = BeautifulSoup(str(div[c]), "html.parser")
            # print(div2.prettify())
            player = div2.find('span',class_='truncate').text
            div3 = str(div2.find('span',class_='flex min-w-0 items-center gap-2'))
            has_svg = fore in str(div2)
            foreigner = 'Yes' if has_svg  else 'No'
            status = 'Retained' if 'Retained' in str(div2) else 'New'
            role = div2.find_all('td',class_='hidden px-4 py-4 tablet:table-cell')[1].text
            has_img = "img" in str(div2)
            capped = 'Yes' if has_img else 'No'
            price =  float(div2.find('td',class_='w-24 px-4 py-4 text-right whitespace-nowrap tablet:w-auto tablet:text-left').text.split()[0][1:])
        
            if c == 0 :
                dictionary = {
                    "sl_no":c+1,
                    'team' : team,
                    'spent(Cr)': spent,
                    'bought':bought,
                    'overseas':overseas,
                    'remaining(Cr)':remaining,
                    'player':player,
                    'foreigner' : foreigner,
                    'status' : status,
                    'role' : role,
                    'capped':capped,
                    'price(Cr)':price,
                }
                
        
            else:
                dictionary = {
                    "sl_no":c+1,
                    'player':player,
                    'foreigner' : foreigner,
                    'status' : status,
                    'role' : role,
                    'capped':capped,
                    'price(Cr)':price,
                }
            players_details.append(dictionary)
            c+=1

# IPL_2026={
#     'team_details':team_details,
#     'players_details':players_details,
# }

# IPL_2026

In [3]:
import pandas as pd


players_df = pd.DataFrame(players_details)
players_df.to_csv("ipl.csv", index=False)
players_df.to_excel("ipl.xlsx", index=False)

df = pd.read_csv('ipl.csv')
df

,sl_no,team,spent(Cr),bought,overseas,remaining(Cr),player,foreigner,status,role,capped,price(Cr)
0,1,Delhi Capitals,15.0,11.0,6.0,0.0,Marizanne Kapp,Yes,Retained,Batter,Yes,2.2
1,2,NaN,NaN,NaN,NaN,NaN,Jemimah Rodrigues,No,Retained,Batter,Yes,2.2
2,3,NaN,NaN,NaN,NaN,NaN,Annabel Sutherland,Yes,Retained,All-Rounder,Yes,2.2
3,4,NaN,NaN,NaN,NaN,NaN,Shafali Verma,No,Retained,Batter,Yes,2.2
4,5,NaN,NaN,NaN,NaN,NaN,Chinelle Henry,Yes,New,Batter,Yes,1.3
...,...,...,...,...,...,...,...,...,...,...,...,...
79,14,NaN,NaN,NaN,NaN,NaN,Shipra Giri,No,New,Wicketkeeper,No,0.1
80,15,NaN,NaN,NaN,NaN,NaN,Suman Meena,No,New,All-Rounder,No,0.1
81,16,NaN,NaN,NaN,NaN,NaN,Tara Norris,Yes,New,Bowler,Yes,0.1
82,17,NaN,NaN,NaN,NaN,NaN,Simran Shaikh,No,New,Batter,No,0.1
